<a href="https://colab.research.google.com/github/Harshavardhan-gowdu-v/Harsha-POC-CODE/blob/main/Multimodel_chat_Google_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pymupdf faiss-cpu pillow sentence-transformers

In [ ]:
import fitz
from google.colab import files
import os

uploaded = files.upload()
file_path = list(uploaded.keys())[0]

doc = fitz.open(file_path)

texts = []
image_paths = []

img_folder = "/content/images"
os.makedirs(img_folder, exist_ok=True)

for page_num in range(len(doc)):
    page = doc[page_num]

    texts.append(page.get_text())

    for i, img in enumerate(page.get_images(full=True)):
        xref = img[0]
        base = doc.extract_image(xref)
        path = f"{img_folder}/p{page_num}_{i}.png"

        with open(path, "wb") as f:
            f.write(base["image"])

        image_paths.append((page_num, path))

print("✅ Extracted:", len(texts), "pages,", len(image_paths), "images")

In [ ]:
chunks = []
chunk_page_map = []

for i, text in enumerate(texts):
    for chunk in text.split("\n\n"):
        if len(chunk) > 100:
            chunks.append(chunk)
            chunk_page_map.append(i)

print("🧩 Chunks:", len(chunks))

In [ ]:
from sentence_transformers import SentenceTransformer

# Text embedding model
text_model = SentenceTransformer('all-MiniLM-L6-v2')

# CLIP for images
clip_model = SentenceTransformer('clip-ViT-B-32')

In [5]:
import numpy as np

text_embeddings = text_model.encode(chunks)

In [ ]:
from PIL import Image

image_embeddings = []
image_files = []

for page, path in image_paths:
    try:
        img = Image.open(path).convert("RGB")
        emb = clip_model.encode(img)

        image_embeddings.append(emb)
        image_files.append(path)
    except:
        continue

image_embeddings = np.array(image_embeddings)

print("🖼 Image embeddings:", len(image_embeddings))

In [7]:
import faiss

# Text index
text_index = faiss.IndexFlatL2(len(text_embeddings[0]))
text_index.add(np.array(text_embeddings))

# Image index
image_index = faiss.IndexFlatL2(len(image_embeddings[0]))
image_index.add(image_embeddings)

In [ ]:
# Build a dictionary: page number → list of image paths
page_to_images = {}

for path in image_files:
    # Extract page number from filename like "p22_0.png"
    filename = os.path.basename(path)
    try:
        page_num = int(filename.split("_")[0].replace("p", ""))
        if page_num not in page_to_images:
            page_to_images[page_num] = []
        page_to_images[page_num].append(path)
    except:
        continue

print(f"✅ Page-to-image mapping built: {len(page_to_images)} pages have images")
for page, imgs in sorted(page_to_images.items())[:5]:
    print(f"  Page {page}: {len(imgs)} images")

In [ ]:
text_embeddings_norm = text_model.encode(chunks, normalize_embeddings=True)
text_index = faiss.IndexFlatIP(text_embeddings_norm.shape[1])
text_index.add(np.array(text_embeddings_norm))
print(f"✅ Text index rebuilt: {text_index.ntotal} vectors (normalized)")

In [33]:
from IPython.display import display, Image as IPImage
import numpy as np
import os

def extract_best_sentences(chunks_list, query, top_n=3):
    all_sentences = []
    for chunk in chunks_list:
        sentences = [s.strip() for s in chunk.replace("\n", " ").split(".") if len(s.strip()) > 20]
        all_sentences.extend(sentences)

    if not all_sentences:
        return "\n".join(chunks_list)

    query_emb = text_model.encode([query], normalize_embeddings=True)
    sent_embs = text_model.encode(all_sentences, normalize_embeddings=True)
    scores = np.dot(sent_embs, query_emb.T).flatten()
    top_indices = np.argsort(scores)[::-1][:top_n]

    best = [all_sentences[i] + "." for i in sorted(top_indices)]
    return "\n".join(best)


def chatbot():
    print("🤖 Multimodal RAG Chatbot\n")

    while True:
        query = input("🧑 Ask: ")

        if query.lower() == "exit":
            break

        # 🔹 TEXT SEARCH
        q_emb = text_model.encode([query], normalize_embeddings=True)
        D, I = text_index.search(np.array(q_emb), k=5)

        best_score = float(D[0][0])
        print(f"[DEBUG] Best text score: {best_score:.3f}")

        # 🔹 OUT-OF-PDF CHECK
        if best_score < 0.35:
            print("\n❌ This question is not related to the document.")
            print("Please ask something related to the PDF content.\n")
            print("=" * 60)
            continue

        # 🔹 GET RELEVANT CHUNKS + PAGES
        retrieved_chunks = []
        retrieved_pages = set()

        for j in range(len(I[0])):
            if float(D[0][j]) > 0.25:
                retrieved_chunks.append(chunks[I[0][j]])
                retrieved_pages.add(chunk_page_map[I[0][j]])

        if not retrieved_chunks:
            print("\n❌ No relevant content found in the document.\n")
            print("=" * 60)
            continue

        # 🔹 ANSWER
        print("\n📄 Answer:\n")
        answer = extract_best_sentences(retrieved_chunks, query, top_n=5)
        print(answer)

        # 🔹 IMAGE RETRIEVAL — based on matching PAGES, not CLIP
        print(f"\n[DEBUG] Retrieved pages: {sorted(retrieved_pages)}")
        print("\n🖼️ Relevant Images:\n")
        shown = 0

        for page in sorted(retrieved_pages):
            if page in page_to_images:
                for path in page_to_images[page]:
                    if os.path.exists(path):
                        print(f"  [Page {page}] {os.path.basename(path)}")
                        display(IPImage(filename=path))
                        shown += 1
                        if shown >= 5:
                            break
            if shown >= 5:
                break

        if shown == 0:
            print("  No images found on the relevant pages.")

        print("\n" + "=" * 60)



In [ ]:
chatbot()

In [ ]:
# Trail and error verification Code

In [17]:
# Rebuild with normalized embeddings
text_embeddings_norm = text_model.encode(chunks, normalize_embeddings=True)
text_index = faiss.IndexFlatIP(text_embeddings_norm.shape[1])
text_index.add(np.array(text_embeddings_norm))
print(f"✅ Text index rebuilt: {text_index.ntotal} vectors (normalized)")

✅ Text index rebuilt: 406 vectors (normalized)


In [18]:
# Normalize existing image embeddings
image_embeddings_norm = image_embeddings / np.linalg.norm(image_embeddings, axis=1, keepdims=True)
image_index = faiss.IndexFlatIP(image_embeddings_norm.shape[1])
image_index.add(image_embeddings_norm)
print(f"✅ Image index rebuilt: {image_index.ntotal} images (normalized)")

✅ Image index rebuilt: 255 images (normalized)


In [19]:
from IPython.display import display, Image as IPImage
import numpy as np
import os

def extract_best_sentences(chunks_list, query, top_n=5):
    all_sentences = []
    for chunk in chunks_list:
        sentences = [s.strip() for s in chunk.replace("\n", " ").split(".") if len(s.strip()) > 20]
        all_sentences.extend(sentences)

    if not all_sentences:
        return "\n".join(chunks_list)

    query_emb = text_model.encode([query], normalize_embeddings=True)
    sent_embs = text_model.encode(all_sentences, normalize_embeddings=True)
    scores = np.dot(sent_embs, query_emb.T).flatten()
    top_indices = np.argsort(scores)[::-1][:top_n]

    best = [all_sentences[i] + "." for i in sorted(top_indices)]
    return "\n".join(best)


def chatbot():
    print("🤖 Multimodal RAG Chatbot\n")

    while True:
        query = input("🧑 Ask: ")

        if query.lower() == "exit":
            break

        # 🔹 TEXT SEARCH
        q_emb = text_model.encode([query], normalize_embeddings=True)
        D, I = text_index.search(np.array(q_emb), k=10)

        best_score = float(D[0][0])
        print(f"[DEBUG] Best text score: {best_score:.3f}")

        # 🔹 OUT-OF-PDF CHECK (cosine scores are now 0 to 1)
        if best_score < 0.35:
            print("\n❌ This question is not related to the document.")
            print("Please ask something related to the PDF content.\n")
            print("=" * 60)
            continue

        # 🔹 GET RELEVANT CHUNKS (only above threshold)
        retrieved_chunks = []
        for j in range(len(I[0])):
            if float(D[0][j]) > 0.25:
                retrieved_chunks.append(chunks[I[0][j]])

        if not retrieved_chunks:
            print("\n❌ No relevant content found in the document.\n")
            print("=" * 60)
            continue

        # 🔹 ANSWER — extract best sentences
        print("\n📄 Answer:\n")
        answer = extract_best_sentences(retrieved_chunks, query, top_n=5)
        print(answer)

        # 🔹 IMAGE SEARCH
        q_img_emb = clip_model.encode([query], normalize_embeddings=True)
        D_img, I_img = image_index.search(np.array(q_img_emb), k=10)

        max_img_score = float(D_img[0][0])
        print(f"\n[DEBUG] Best image score: {max_img_score:.3f}")

        print("\n🖼️ Relevant Images:\n")
        shown = 0

        for score, idx in zip(D_img[0], I_img[0]):
            if idx < len(image_files):
                path = image_files[idx]
                if float(score) > 0.20 and os.path.exists(path):
                    print(f"  [Score: {score:.3f}] {os.path.basename(path)}")
                    display(IPImage(filename=path))
                    shown += 1
                    if shown >= 3:
                        break

        if shown == 0:
            print("  No relevant images found for this query.")

        print("\n" + "=" * 60)



In [ ]:
chatbot()